# NB02: exploratory data analysis

**Purpose:** describe the acquired sample and inspect draft-slot outcome patterns without making inferential claims.

**Pipeline:** load analysis panel -> summarize coverage and scoring -> summarize outcomes by slot -> save EDA tables.

**Inputs:** `data/processed/analysis_panel.csv`. **Outputs:** `artifacts/eda_season_coverage.csv` and `artifacts/eda_slot_summary.csv`.

**Run:** execute after NB01.

**Locked definitions:** all summaries are descriptive for the public convenience sample. Each draft slot has one observation per retained league-season.

**Gate table:** equal sample size across slots; season totals sum to the panel total; no non-finite outcome values.


This cell loads the prepared panel and confirms balanced exposure by slot.


In [1]:
# CELL [1 load-and-coverage]
import csv, math
from collections import Counter
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
with (ROOT / 'data/processed/analysis_panel.csv').open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    for field in ['season', 'draft_slot']:
        row[field] = int(row[field])
    for field in ['regular_season_points', 'points_rank', 'points_zscore', 'top_6_points', 'top_regular_season_scorer']:
        row[field] = float(row[field])
slot_n = Counter(r['draft_slot'] for r in rows)
if len(slot_n) != 12 or len(set(slot_n.values())) != 1 or not all(math.isfinite(r['points_zscore']) for r in rows):
    raise RuntimeError('EDA gate failed')
season_n = Counter(r['season'] for r in rows)
print({'team_seasons': len(rows), 'league_seasons': len(rows) // 12, 'per_slot_n': dict(sorted(slot_n.items())), 'season_team_rows': dict(sorted(season_n.items()))})


{'team_seasons': 43692, 'league_seasons': 3641, 'per_slot_n': {1: 3641, 2: 3641, 3: 3641, 4: 3641, 5: 3641, 6: 3641, 7: 3641, 8: 3641, 9: 3641, 10: 3641, 11: 3641, 12: 3641}, 'season_team_rows': {2018: 108, 2019: 1608, 2020: 4404, 2021: 6036, 2022: 6312, 2023: 7212, 2024: 9144, 2025: 8868}}


Each draft slot has exactly 3,641 observations. Coverage now spans 2018 to 2025; 2024 and 2025 together supply 41.2% of league-seasons, while the early historic seasons are thinner. Exposure is equal by slot, but season robustness remains necessary.


This cell writes the season coverage table. It is the primary check on the temporal composition of the public network sample.


In [2]:
# CELL [2 season-coverage]
coverage = [{'season': season, 'team_seasons': count, 'league_seasons': count // 12, 'share_of_league_seasons': (count // 12) / (len(rows) // 12)} for season, count in sorted(season_n.items())]
with (ROOT / 'artifacts/eda_season_coverage.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(coverage[0])); writer.writeheader(); writer.writerows(coverage)
coverage


[{'season': 2018,
  'team_seasons': 108,
  'league_seasons': 9,
  'share_of_league_seasons': 0.0024718483932985444},
 {'season': 2019,
  'team_seasons': 1608,
  'league_seasons': 134,
  'share_of_league_seasons': 0.03680307607800055},
 {'season': 2020,
  'team_seasons': 4404,
  'league_seasons': 367,
  'share_of_league_seasons': 0.10079648448228509},
 {'season': 2021,
  'team_seasons': 6036,
  'league_seasons': 503,
  'share_of_league_seasons': 0.13814886020324088},
 {'season': 2022,
  'team_seasons': 6312,
  'league_seasons': 526,
  'share_of_league_seasons': 0.14446580609722603},
 {'season': 2023,
  'team_seasons': 7212,
  'league_seasons': 601,
  'share_of_league_seasons': 0.16506454270804724},
 {'season': 2024,
  'team_seasons': 9144,
  'league_seasons': 762,
  'share_of_league_seasons': 0.20928316396594343},
 {'season': 2025,
  'team_seasons': 8868,
  'league_seasons': 739,
  'share_of_league_seasons': 0.20296621807195825}]

The expanded sample is 0.2% from 2018, 3.7% from 2019, 10.1% from 2020, 13.8% from 2021, and 72.2% from 2022 to 2025. The historic extension materially broadens the window, but recent seasons still dominate the pooled result.


This cell summarizes regular-season outcomes by draft slot, including the two requested success benchmarks.


In [3]:
# CELL [3 slot-outcomes]
summary = []
for slot in range(1, 13):
    x = [r for r in rows if r['draft_slot'] == slot]
    n = len(x)
    summary.append({'draft_slot': slot, 'n': n, 'mean_points_zscore': sum(r['points_zscore'] for r in x) / n, 'mean_points_rank': sum(r['points_rank'] for r in x) / n, 'top_6_rate': sum(r['top_6_points'] for r in x) / n, 'top_scorer_rate': sum(r['top_regular_season_scorer'] for r in x) / n})
with (ROOT / 'artifacts/eda_slot_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0])); writer.writeheader(); writer.writerows(summary)
summary


[{'draft_slot': 1,
  'n': 3641,
  'mean_points_zscore': -0.10245139740133252,
  'mean_points_rank': 6.829717110683878,
  'top_6_rate': 0.4605877506179621,
  'top_scorer_rate': 0.07047056669413165},
 {'draft_slot': 2,
  'n': 3641,
  'mean_points_zscore': -0.0060861025223603316,
  'mean_points_rank': 6.480911837407306,
  'top_6_rate': 0.49766547651744025,
  'top_scorer_rate': 0.07898471115993776},
 {'draft_slot': 3,
  'n': 3641,
  'mean_points_zscore': 0.039445789167185476,
  'mean_points_rank': 6.3575940675638565,
  'top_6_rate': 0.5146937654490524,
  'top_scorer_rate': 0.08827703011993043},
 {'draft_slot': 4,
  'n': 3641,
  'mean_points_zscore': 0.056020663262519095,
  'mean_points_rank': 6.287833012908542,
  'top_6_rate': 0.5280142817907169,
  'top_scorer_rate': 0.09381580151972903},
 {'draft_slot': 5,
  'n': 3641,
  'mean_points_zscore': 0.030074398297769898,
  'mean_points_rank': 6.388904147212305,
  'top_6_rate': 0.5126338917879704,
  'top_scorer_rate': 0.08420305776801246},
 {'dra

Slot 4 leads every descriptive outcome: +0.056 points z-score, 52.8% top-six rate, and 9.38% top-scorer rate. Slot 1 is lowest on all three: -0.102, 46.1%, and 7.05%. These are descriptive patterns; NB03 tests their precision.


### What this cell does

- Defines the reusable Plotly bar-chart helper.
- Renders the first of five interactive EDA charts: league coverage by year.


In [4]:
# CELL [4 coverage-chart]
import plotly.graph_objects as go
def plotly_bar(path, title, labels, values, baseline=None, color='#3367d6', y_title='Value'):
    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=color, text=[f'{value:.3g}' for value in values], textposition='outside'))
    if baseline is not None:
        fig.add_hline(y=baseline, line_dash='dash', line_color='#d93025', annotation_text='baseline')
    fig.update_layout(title=title, template='plotly_white', height=460, xaxis_title='Draft slot or season', yaxis_title=y_title, showlegend=False)
    fig.write_html(path.with_suffix('.html'), include_plotlyjs='cdn')
    fig.show()
    return fig
svg_bar = plotly_bar  # Backward-compatible name for the following chart cells.
fig_coverage = plotly_bar(ROOT / 'artifacts/eda_01_coverage_by_season.html', 'League-seasons by year', [str(x['season']) for x in coverage], [x['league_seasons'] for x in coverage], color='#5f6368', y_title='League-seasons')



### Interpreting the output

- 2024 has the largest coverage at 762 league-seasons; 2018 has only 9 and is useful as context, not as a standalone estimate.
- The 2020 onward seasons each have at least 367 league-seasons, which supports a more credible year-by-year check.
- The visible imbalance is why NB03 checks the headline contrast within season.
- It does not show scoring differences.


### What this cell does

- Plots standardized regular-season points by draft slot.


In [5]:
# CELL [5 zscore-chart]
svg_bar(ROOT / 'artifacts/eda_02_zscore_by_slot.svg', 'Mean regular-season points within league', [str(x['draft_slot']) for x in summary], [x['mean_points_zscore'] for x in summary], baseline=0.0);


### Interpreting the output

- Slot 4 has the highest mean within-league z-score at +0.056, while slot 1 is lowest at -0.102.
- Slots 3 through 8 are all above average, with no second discontinuity as large as the first-slot deficit.
- This 0.158 z-score spread between slots 4 and 1 motivates the clustered intervals in NB03.
- It does not establish precision or causality.


### What this cell does

- Plots mean regular-season points rank by draft slot.


In [6]:
# CELL [6 rank-chart]
svg_bar(ROOT / 'artifacts/eda_03_rank_by_slot.svg', 'Mean regular-season points rank by draft slot', [str(x['draft_slot']) for x in summary], [x['mean_points_rank'] for x in summary], baseline=6.5, color='#7b1fa2');


### Interpreting the output

- Slot 4 has the best average rank at 6.29 and slot 1 the worst at 6.83.
- This agrees with the z-score chart rather than contradicting it.
- The rank result confirms the finding is not an artifact of a few extreme point totals.
- It does not quantify uncertainty.


### What this cell does

- Plots the requested top-six regular-season scoring rate by slot.


In [7]:
# CELL [7 top-six-chart]
svg_bar(ROOT / 'artifacts/eda_04_top_six_by_slot.svg', 'Top-six regular-season scorer rate', [str(x['draft_slot']) for x in summary], [x['top_6_rate'] for x in summary], baseline=0.5, color='#188038');


### Interpreting the output

- Slot 4 reaches the top six in 52.8% of league-seasons, 2.80 points above the 50% baseline.
- Slot 1 reaches the top six 46.1% of the time, 3.94 points below baseline.
- The observed slot-4 minus slot-1 gap is 6.74 points, the main practical contrast for NB03.
- It does not show whether a difference survives sampling variation.


### What this cell does

- Plots the requested outright regular-season top-scorer rate by slot.


In [8]:
# CELL [8 top-scorer-chart]
svg_bar(ROOT / 'artifacts/eda_05_top_scorer_by_slot.svg', 'Outright regular-season top-scorer rate', [str(x['draft_slot']) for x in summary], [x['top_scorer_rate'] for x in summary], baseline=1/12, color='#e37400');


### Interpreting the output

- Slot 4 leads at 9.38%, 1.05 points above the 8.33% baseline; slot 1 is lowest at 7.05%.
- The slot-4 minus slot-1 spread is 2.33 points on the rarest outcome.
- The direction agrees with the other outcomes, but uncertainty matters more here because only one top scorer exists per league-season.
- It does not prove a slot advantage.


### What this cell does

- Uses Plotly to show draft-slot points patterns by season.
- The left panel compares every available season; the right panel isolates the latest three seasons.


In [9]:
# CELL [9 plotly-year-over-year-points]
import plotly.graph_objects as go
from plotly.subplots import make_subplots
years = sorted({r['season'] for r in rows})
recent_years = years[-3:]
def z_by_slot(subset):
    return [sum(r['points_zscore'] for r in subset if r['draft_slot'] == slot) / sum(r['draft_slot'] == slot for r in subset) for slot in range(1, 13)]
fig = make_subplots(rows=1, cols=2, subplot_titles=('All available seasons', f'Latest three seasons: {recent_years[0]} to {recent_years[-1]}'))
for year in years:
    fig.add_trace(go.Scatter(x=list(range(1, 13)), y=z_by_slot([r for r in rows if r['season'] == year]), mode='lines+markers', name=str(year), legendgroup=str(year)), row=1, col=1)
for year in recent_years:
    fig.add_trace(go.Scatter(x=list(range(1, 13)), y=z_by_slot([r for r in rows if r['season'] == year]), mode='lines+markers', name=str(year), legendgroup=str(year), showlegend=False), row=1, col=2)
for column in [1, 2]: fig.add_hline(y=0, line_dash='dash', line_color='firebrick', row=1, col=column)
fig.update_layout(title='Draft slot and regular-season points shift across years', template='plotly_white', height=520, legend_title='Season')
fig.update_xaxes(title_text='Draft slot', dtick=1); fig.update_yaxes(title_text='Mean within-league points z-score')
fig.write_html(ROOT / 'artifacts/eda_06_year_over_year_points.html', include_plotlyjs='cdn')
fig.show()


### Interpreting the output

- The 2018 and 2019 lines are visibly volatile because they contain only 9 and 134 leagues. The 2020 onward lines are the useful stability check.
- In the latest-three-season panel, slot 4 remains generally above zero while slot 1 is below zero in two of the three seasons, but the remaining slots cross frequently.
- This supports a modest pooled contrast rather than a clean monotonic draft-order advantage.
- It does not replace the cluster-aware inference in NB03.


## Conclusion

The EDA now covers the expanded historical panel and adds an interactive Plotly year-over-year draft-slot points view. NB03 quantifies uncertainty and checks robustness.
